# AI Programming — Lecture 11
## Lab 2-2: ETTh1 Univariate Direct Forecasting

과거 96시간의 **Oil Temperature (`OT`)**만 사용하여 이후 24시간의 `OT`를 직접 예측합니다.

```text
Past OT: 96 steps
       ↓
      MLP
       ↓
Future OT: 24 steps
```

### 학습 목표
- time-series forecasting에서 chronological split이 필요한 이유를 설명할 수 있습니다.
- train 구간에만 scaler를 fitting할 수 있습니다.
- sliding window로 `(past, future)` sample을 만들 수 있습니다.
- Last Value / Seasonal Naive baseline을 구현할 수 있습니다.
- MLP가 미래 24개 값을 한 번에 예측하는 **direct forecasting**을 이해할 수 있습니다.
- naive baseline과 MLP의 성능을 MSE/MAE로 비교할 수 있습니다.

### Colab 실행 안내
원본 실험은 여러 model과 random seed를 반복 비교할 수 있도록 구성되어 있습니다.
학생 실습에서는 Colab 실행 시간을 고려하여 기본값을 다음처럼 줄였습니다.

- `MAX_EPOCHS = 150`
- `BATCH_SIZE = 64`
- 기본적으로 repeated-seed confirmation은 수행하지 않음
- 모든 실험에는 early stopping 적용

필요하면 `RUN_REPEATED_CONFIRMATION=True`로 바꾸어 추가 실험할 수 있습니다.

## 1. 실습 환경 설정

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input,
    Flatten,
    Dense,
    Dropout,
    BatchNormalization,
    Activation,
)
from tensorflow.keras.callbacks import EarlyStopping

INPUT_LEN = 96
PRED_LEN = 24
BATCH_SIZE = 64
MAX_EPOCHS = 150
LEARNING_RATE = 1e-3

SCREENING_SEED = 42
REPEAT_SEEDS = [42, 123, 2026]
TOP_K = 2
RUN_REPEATED_CONFIRMATION = False
VERBOSE = 1

def reset_random_state(seed):
    tf.keras.backend.clear_session()
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

reset_random_state(SCREENING_SEED)

print("TensorFlow version:", tf.__version__)

# Colab runtime 확인
gpus = tf.config.list_physical_devices('GPU')
print('GPU:', gpus[0].name if gpus else '사용하지 않음 (CPU)')

## 2. ETTh1 데이터 불러오기

ETTh1은 transformer의 operating condition을 시간 단위로 기록한 시계열 데이터입니다.

이번 실습에서는 `OT`만 사용합니다.

데이터 파일을 다음 위치에 두세요.

```text
MyDrive/Colab Notebooks/data/ETTh1.csv
```

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Google Colab이 아닙니다. 로컬 경로를 FILE_PATH에 지정하세요.")

FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/data/ETTh1.csv"

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(
        f"데이터 파일을 찾을 수 없습니다: {FILE_PATH}\n"
        "FILE_PATH를 실제 ETTh1.csv 위치로 수정하세요."
    )

df = pd.read_csv(FILE_PATH)
df["date"] = pd.to_datetime(df["date"])

print("Data shape:", df.shape)
print("Date range:", df["date"].min(), "to", df["date"].max())
display(df[["date", "OT"]].head())

## 3. Chronological Split, Scaling, Window 생성

시간 순서를 유지한 채 다음과 같이 분할합니다.

```text
Train      70%
Validation 15%
Test       15%
```

Scaler는 **train 구간의 OT에만 fit**합니다.

Validation/Test의 첫 forecast는 해당 split 직전의 과거 96시간을 context로 사용할 수 있도록,
전체 시계열에서 먼저 window를 만든 뒤 **future target 위치**를 기준으로 split을 결정합니다.

In [ ]:
FEATURE_COLS = ["OT"]
TARGET_COL = "OT"
TARGET_FEATURE_INDEX = 0

X_raw = df[FEATURE_COLS].to_numpy(dtype=np.float32)
y_raw = df[[TARGET_COL]].to_numpy(dtype=np.float32)

n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

x_scaler = StandardScaler()
y_scaler = StandardScaler()

x_scaler.fit(X_raw[:train_end])
y_scaler.fit(y_raw[:train_end])

X_scaled = x_scaler.transform(X_raw).astype(np.float32)
y_scaled = y_scaler.transform(y_raw).astype(np.float32)


def create_all_windows(features, target, input_len, pred_len):
    X, y, target_starts, target_ends = [], [], [], []

    max_start = len(features) - input_len - pred_len + 1

    for input_start in range(max_start):
        target_start = input_start + input_len
        target_end = target_start + pred_len

        X.append(features[input_start:target_start])
        y.append(target[target_start:target_end, 0])
        target_starts.append(target_start)
        target_ends.append(target_end)

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.float32),
        np.asarray(target_starts),
        np.asarray(target_ends),
    )


X_all, y_all, target_starts, target_ends = create_all_windows(
    X_scaled,
    y_scaled,
    INPUT_LEN,
    PRED_LEN,
)

train_mask = target_ends <= train_end
val_mask = (target_starts >= train_end) & (target_ends <= val_end)
test_mask = (target_starts >= val_end) & (target_ends <= n)

X_train, y_train = X_all[train_mask], y_all[train_mask]
X_val, y_val = X_all[val_mask], y_all[val_mask]
X_test, y_test = X_all[test_mask], y_all[test_mask]

window_summary = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "X Shape": [str(X_train.shape), str(X_val.shape), str(X_test.shape)],
    "y Shape": [str(y_train.shape), str(y_val.shape), str(y_test.shape)],
})

display(window_summary)

## 4. Naive Baseline

신경망이 반드시 단순한 baseline보다 좋아야 하는 것은 아닙니다.

두 baseline을 먼저 계산합니다.

### Last Value
마지막 관측값을 미래 24시간 동안 그대로 유지합니다.

### Seasonal Naive (24h)
직전 24시간의 패턴을 다음 24시간 예측으로 그대로 사용합니다.

In [ ]:
def inverse_target(values_scaled):
    original_shape = values_scaled.shape
    return y_scaler.inverse_transform(
        values_scaled.reshape(-1, 1)
    ).reshape(original_shape)


def evaluate_predictions(y_true_scaled, y_pred_scaled):
    y_true_actual = inverse_target(y_true_scaled)
    y_pred_actual = inverse_target(y_pred_scaled)

    return {
        "MSE (Normalized)": mean_squared_error(
            y_true_scaled.ravel(), y_pred_scaled.ravel()
        ),
        "MAE (Normalized)": mean_absolute_error(
            y_true_scaled.ravel(), y_pred_scaled.ravel()
        ),
        "MSE (°C²)": mean_squared_error(
            y_true_actual.ravel(), y_pred_actual.ravel()
        ),
        "MAE (°C)": mean_absolute_error(
            y_true_actual.ravel(), y_pred_actual.ravel()
        ),
    }


def make_naive_predictions(X):
    last_value = X[:, -1, 0]
    last_value_pred = np.repeat(last_value[:, None], PRED_LEN, axis=1)
    seasonal_pred = X[:, -PRED_LEN:, 0]

    return {
        "Last Value": last_value_pred,
        "Seasonal Naive (24h)": seasonal_pred,
    }


val_baseline_records = []

for name, pred in make_naive_predictions(X_val).items():
    row = {"Model": name}
    row.update(evaluate_predictions(y_val, pred))
    val_baseline_records.append(row)

display(
    pd.DataFrame(val_baseline_records).style.format({
        "MSE (Normalized)": "{:.4f}",
        "MAE (Normalized)": "{:.4f}",
        "MSE (°C²)": "{:.4f}",
        "MAE (°C)": "{:.4f}",
    })
)

## 5. MLP와 Training 함수

모델 구조:

```text
96
→ Flatten
→ Dense(128, ReLU)
→ Dense(64, ReLU)
→ Dense(24)
```

출력 24개가 미래 24시간을 **한 번에 직접 예측**합니다.

In [ ]:
def add_dense_block(layers, units, use_batch_norm=False, dropout_rate=0.0):
    layers.append(
        Dense(
            units,
            kernel_initializer="he_normal",
            use_bias=not use_batch_norm,
        )
    )

    if use_batch_norm:
        layers.append(BatchNormalization())

    layers.append(Activation("relu"))

    if dropout_rate > 0:
        layers.append(Dropout(dropout_rate))


def build_mlp(config):
    layers = [
        Input(shape=(INPUT_LEN, 1)),
        Flatten(),
    ]

    add_dense_block(
        layers,
        128,
        use_batch_norm=config.get("use_batch_norm", False),
        dropout_rate=config.get("dropout_rate", 0.0),
    )
    add_dense_block(
        layers,
        64,
        use_batch_norm=config.get("use_batch_norm", False),
        dropout_rate=config.get("dropout_rate", 0.0),
    )
    layers.append(Dense(PRED_LEN))

    model = Sequential(layers)

    if config.get("optimizer", "adam").lower() == "adamw":
        optimizer = tf.keras.optimizers.AdamW(
            learning_rate=LEARNING_RATE,
            weight_decay=config.get("weight_decay", 1e-4),
        )
    else:
        optimizer = tf.keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        )

    model.compile(
        optimizer=optimizer,
        loss="mse",
        metrics=["mae"],
    )

    return model


MODEL_DIR = Path("/content/etth1_univariate_ot_models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)


def fit_for_validation(config, seed, save_model=True):
    reset_random_state(seed)
    model = build_mlp(config)

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        shuffle=config.get("shuffle", True),
        callbacks=[
            EarlyStopping(
                monitor="val_loss",
                patience=15,
                restore_best_weights=True,
                mode="min",
                verbose=1,
            )
        ],
        verbose=VERBOSE,
    )

    best_index = int(np.argmin(history.history["val_loss"]))

    record = {
        "Experiment": config["name"],
        "Seed": seed,
        "Parameters": model.count_params(),
        "Best Epoch": best_index + 1,
        "Best Val Loss": float(history.history["val_loss"][best_index]),
        "Best Val MAE (°C)": float(
            history.history["val_mae"][best_index] * y_scaler.scale_[0]
        ),
    }

    model_path = None

    if save_model:
        safe_name = "".join(
            ch if ch.isalnum() else "_" for ch in config["name"]
        ).strip("_")
        model_path = MODEL_DIR / f"{safe_name}_seed{seed}.keras"
        model.save(model_path, overwrite=True)

    return record, str(model_path) if model_path else None

## 6. 참고 실험

같은 MLP 구조를 기반으로 다음 구성을 비교할 수 있습니다.

1. Basic MLP (`shuffle=False`)
2. Basic MLP + Shuffle
3. BatchNorm
4. AdamW
5. BatchNorm + AdamW
6. Dropout + AdamW

Lecture 11의 기본 direct forecasting을 확인할 때는 첫 번째 모델에 특히 주목하세요.

In [ ]:
EXPERIMENTS = [
    {
        "name": "1. Basic MLP (No Shuffle)",
        "shuffle": False,
        "optimizer": "adam",
    },
    {
        "name": "2. Basic MLP + Shuffle",
        "shuffle": True,
        "optimizer": "adam",
    },
    {
        "name": "3. MLP + BatchNorm",
        "shuffle": True,
        "use_batch_norm": True,
        "optimizer": "adam",
    },
    {
        "name": "4. MLP + AdamW",
        "shuffle": True,
        "optimizer": "adamw",
        "weight_decay": 1e-4,
    },
    {
        "name": "5. BatchNorm + AdamW",
        "shuffle": True,
        "use_batch_norm": True,
        "optimizer": "adamw",
        "weight_decay": 1e-4,
    },
    {
        "name": "6. Dropout + AdamW",
        "shuffle": True,
        "dropout_rate": 0.1,
        "optimizer": "adamw",
        "weight_decay": 1e-4,
    },
]

config_by_name = {config["name"]: config for config in EXPERIMENTS}

## 7. Single-Seed Screening

Colab 실습에서는 먼저 seed 하나로 각 설정을 빠르게 비교합니다.

모든 실험을 완전히 재현하기보다 **구성에 따른 상대적 차이**를 확인하는 것이 목적입니다.

In [ ]:
screening_records = []
screening_paths = {}

for config in EXPERIMENTS:
    print("\n" + "=" * 80)
    print(config["name"])

    record, model_path = fit_for_validation(
        config,
        seed=SCREENING_SEED,
        save_model=True,
    )

    screening_records.append(record)
    screening_paths[config["name"]] = model_path

screening_df = pd.DataFrame(screening_records).sort_values(
    "Best Val Loss"
).reset_index(drop=True)

display(
    screening_df.style.format({
        "Parameters": "{:,}",
        "Best Val Loss": "{:.4f}",
        "Best Val MAE (°C)": "{:.4f}",
    }).highlight_min(
        subset=["Best Val Loss", "Best Val MAE (°C)"],
        axis=0,
    )
)

top_names = screening_df.head(TOP_K)["Experiment"].tolist()
print("Repeated-seed candidates:", top_names)

## 8. 선택 실습 — Repeated-Seed Confirmation

기본값에서는 실행하지 않습니다.

`RUN_REPEATED_CONFIRMATION=True`로 바꾸면 상위 설정을 여러 random seed에서 다시 학습합니다.
수업 시간에는 생략해도 됩니다.

In [ ]:
repeat_records = []
repeat_paths = {}

if RUN_REPEATED_CONFIRMATION:
    for experiment_name in top_names:
        config = config_by_name[experiment_name]

        for seed in REPEAT_SEEDS:
            print("\n" + "=" * 80)
            print(experiment_name, "| seed =", seed)

            record, model_path = fit_for_validation(
                config,
                seed=seed,
                save_model=True,
            )

            repeat_records.append(record)
            repeat_paths[(experiment_name, seed)] = model_path

    repeat_df = pd.DataFrame(repeat_records)

    repeat_summary = (
        repeat_df
        .groupby("Experiment", as_index=False)
        .agg(
            Val_Loss_Mean=("Best Val Loss", "mean"),
            Val_Loss_Std=("Best Val Loss", "std"),
            Val_MAE_C_Mean=("Best Val MAE (°C)", "mean"),
            Val_MAE_C_Std=("Best Val MAE (°C)", "std"),
        )
        .sort_values("Val_Loss_Mean")
        .reset_index(drop=True)
    )

    display(
        repeat_summary.style.format({
            "Val_Loss_Mean": "{:.4f}",
            "Val_Loss_Std": "{:.4f}",
            "Val_MAE_C_Mean": "{:.4f}",
            "Val_MAE_C_Std": "{:.4f}",
        }).highlight_min(
            subset=["Val_Loss_Mean", "Val_MAE_C_Mean"],
            axis=0,
        )
    )

    selected_experiment = repeat_summary.iloc[0]["Experiment"]
else:
    repeat_df = pd.DataFrame()
    repeat_summary = pd.DataFrame()
    selected_experiment = screening_df.iloc[0]["Experiment"]

print("Selected univariate experiment:")
print(selected_experiment)

## 9. Test 성능과 Naive Baseline 비교

In [ ]:
def evaluate_saved_model(model_path):
    model = tf.keras.models.load_model(model_path)
    pred = model.predict(X_test, verbose=0)
    return model, pred, evaluate_predictions(y_test, pred)


test_records = []
predictions = {}
models = {}

if RUN_REPEATED_CONFIRMATION:
    selected_rows = repeat_df[
        repeat_df["Experiment"] == selected_experiment
    ]

    for _, row in selected_rows.iterrows():
        seed = int(row["Seed"])
        model_path = repeat_paths[(selected_experiment, seed)]

        model, pred, metrics = evaluate_saved_model(model_path)

        test_records.append({
            "Model": selected_experiment,
            "Seed": seed,
            **metrics,
        })
        predictions[seed] = pred
        models[seed] = model

    selected_test_df = pd.DataFrame(test_records)
else:
    model_path = screening_paths[selected_experiment]
    model, pred, metrics = evaluate_saved_model(model_path)

    selected_test_df = pd.DataFrame([{
        "Model": selected_experiment,
        "Seed": SCREENING_SEED,
        **metrics,
    }])
    predictions[SCREENING_SEED] = pred
    models[SCREENING_SEED] = model

display(
    selected_test_df.style.format({
        "MSE (Normalized)": "{:.4f}",
        "MAE (Normalized)": "{:.4f}",
        "MSE (°C²)": "{:.4f}",
        "MAE (°C)": "{:.4f}",
    })
)

baseline_records = []

for name, pred in make_naive_predictions(X_test).items():
    row = {"Model": name, "Type": "Baseline"}
    row.update(evaluate_predictions(y_test, pred))
    baseline_records.append(row)

model_summary_row = {
    "Model": selected_experiment,
    "Type": "Selected Univariate MLP",
    "MSE (Normalized)": selected_test_df["MSE (Normalized)"].mean(),
    "MAE (Normalized)": selected_test_df["MAE (Normalized)"].mean(),
    "MSE (°C²)": selected_test_df["MSE (°C²)"].mean(),
    "MAE (°C)": selected_test_df["MAE (°C)"].mean(),
}

final_comparison_df = pd.concat(
    [
        pd.DataFrame(baseline_records),
        pd.DataFrame([model_summary_row]),
    ],
    ignore_index=True,
)

display(
    final_comparison_df.style.format({
        "MSE (Normalized)": "{:.4f}",
        "MAE (Normalized)": "{:.4f}",
        "MSE (°C²)": "{:.4f}",
        "MAE (°C)": "{:.4f}",
    }).highlight_min(
        subset=[
            "MSE (Normalized)",
            "MAE (Normalized)",
            "MSE (°C²)",
            "MAE (°C)",
        ],
        axis=0,
    )
)

## 10. Forecast 예제 시각화

첫 test sample에 대해

- 과거 96시간
- 실제 미래 24시간
- MLP 예측 미래 24시간

을 한 그림에서 비교합니다.

In [ ]:
if RUN_REPEATED_CONFIRMATION:
    representative_row = (
        repeat_df[
            repeat_df["Experiment"] == selected_experiment
        ]
        .sort_values("Best Val Loss")
        .iloc[0]
    )
    representative_seed = int(representative_row["Seed"])
else:
    representative_seed = SCREENING_SEED

pred_actual = inverse_target(predictions[representative_seed])
y_test_actual = inverse_target(y_test)

history_ot = (
    X_test[0, :, 0] * x_scaler.scale_[0]
    + x_scaler.mean_[0]
)

past_steps = np.arange(-INPUT_LEN, 0)
future_steps = np.arange(PRED_LEN)

plt.figure(figsize=(11, 5))
plt.plot(past_steps, history_ot, label="Historical OT")
plt.plot(
    future_steps,
    y_test_actual[0],
    marker="o",
    label="Actual Future",
)
plt.plot(
    future_steps,
    pred_actual[0],
    marker="o",
    label="Predicted Future",
)
plt.axvline(0, linestyle="--")
plt.xlabel("Time Step")
plt.ylabel("Oil Temperature (°C)")
plt.title(
    f"Univariate 24-Hour Forecast: {selected_experiment}"
)
plt.legend()
plt.grid(True)
plt.show()

## 11. 선택 실습 — Multivariate 결과와 비교

다변량 실험 결과 CSV가 있다면 단변량 결과와 비교할 수 있습니다.

단, 비교하려면 split / input length / prediction length가 동일해야 합니다.

In [ ]:
MULTIVARIATE_RESULTS_PATH = (
    "/content/drive/MyDrive/Colab Notebooks/models/"
    "etth1_mlp_multivariate_revised/"
    "final_comparison_with_baselines.csv"
)

if os.path.exists(MULTIVARIATE_RESULTS_PATH):
    multivariate_df = pd.read_csv(MULTIVARIATE_RESULTS_PATH)

    multivariate_mlp = multivariate_df[
        multivariate_df["Type"] == "Selected MLP"
    ].copy()
    multivariate_mlp["Input"] = "7 variables"

    univariate_mlp = pd.DataFrame([model_summary_row])
    univariate_mlp["Input"] = "OT only"

    input_comparison_df = pd.concat(
        [univariate_mlp, multivariate_mlp],
        ignore_index=True,
    )

    display(
        input_comparison_df[[
            "Input",
            "Model",
            "MSE (Normalized)",
            "MAE (Normalized)",
            "MSE (°C²)",
            "MAE (°C)",
        ]].style.format({
            "MSE (Normalized)": "{:.4f}",
            "MAE (Normalized)": "{:.4f}",
            "MSE (°C²)": "{:.4f}",
            "MAE (°C)": "{:.4f}",
        })
    )
else:
    print(
        "다변량 결과 CSV를 찾지 못했습니다. "
        "다변량 노트북을 먼저 실행하거나 경로를 수정하세요."
    )

## 12. 결과와 Model 저장

In [ ]:
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Colab Notebooks/models/"
    "etth1_mlp_univariate_ot"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

screening_csv = OUTPUT_DIR / "univariate_screening_results.csv"
test_csv = OUTPUT_DIR / "univariate_selected_test_results.csv"
comparison_csv = OUTPUT_DIR / "univariate_final_comparison.csv"
selected_model_path = OUTPUT_DIR / "selected_univariate_ot_mlp.keras"

screening_df.to_csv(screening_csv, index=False)
selected_test_df.to_csv(test_csv, index=False)
final_comparison_df.to_csv(comparison_csv, index=False)

if RUN_REPEATED_CONFIRMATION:
    repeat_df.to_csv(
        OUTPUT_DIR / "univariate_repeated_validation_results.csv",
        index=False,
    )
    repeat_summary.to_csv(
        OUTPUT_DIR / "univariate_repeated_validation_summary.csv",
        index=False,
    )

models[representative_seed].save(
    selected_model_path,
    overwrite=True,
)

print("Saved model:", selected_model_path)
print("Saved results:", comparison_csv)

## 13. 결과 해석 체크리스트

1. Last Value와 Seasonal Naive 중 어떤 baseline이 더 강합니까?
2. Basic MLP가 두 baseline을 모두 이깁니까?
3. MLP prediction이 실제 future보다 지나치게 smooth하지는 않습니까?
4. `shuffle=False`와 `shuffle=True`의 차이는 어떻습니까?
5. 단순한 last-value baseline이 강하다면 OT 시계열의 어떤 특성을 의미합니까?
6. MLP가 sharp change를 예측하기 어려운 이유는 무엇일까요?

> 복잡한 model보다 강한 baseline을 먼저 확인하는 습관이 중요합니다.